# PDF のページを画像にして視覚言語モデルで読む（表とグラフ）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/30_vision_language/pdf_vlm.ipynb)

NLP クックブック 30-05 ／ 画像と言語

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

表やグラフを含む PDF を読み取りたい。テキストを抽出するだけ（データの収集と入出力の章の「PDF からテキストを抽出する (PyMuPDF)」）では表の行と列の対応やグラフの値が失われるので、ページを画像にして視覚言語モデル (VLM) の Qwen3-VL-4B-Instruct に読ませ、テキストだけを渡した場合と比べる。

In [1]:
!pip install -q pymupdf matplotlib-fontja

In [2]:
import urllib.request
import pymupdf  # PyMuPDF (import fitz と同じ)

# 出典:「統計トピックスNo.148 我が国のこどもの数」（総務省統計局）（https://www.stat.go.jp/data/jinsui/topics/topi1480.html）を加工して作成
# 利用条件: 公共データ利用規約 第1.0版 (PDL1.0)。CC BY 4.0 と互換
urllib.request.urlretrieve("https://www.stat.go.jp/data/jinsui/topics/pdf/topics148.pdf", "topics148.pdf")
pub = pymupdf.open("topics148.pdf")[1]      # 2 ページ目: 表 1、図 1 (棒グラフ)、表 2 (見出しが 2 段の表)
text = pub.get_text()
print(len(text), "文字")
print(text[:420].replace("\n", " | "))      # 表 2 の部分。見出しの語順が崩れ、数値は 1 列に並ぶ

# スキャンした PDF を模して、ページの画像だけを貼った PDF を作ると、テキスト層がないので何も取れない
scan = pymupdf.open()
scan.new_page(width=pub.rect.width, height=pub.rect.height).insert_image(pub.rect, pixmap=pub.get_pixmap(dpi=100))
print("画像だけの PDF から抽出したテキスト:", repr(scan[0].get_text()))

1168 文字
- 2 -  |   | 注）表中の数値は、単位未満を四捨五入しているため、  | 合計の数値と内訳の計が一致しない場合があります  | （以下同じ。）。  | こどもの未就学の乳幼児(０～５歳) | 小学生(６～11歳) | 中学生 | 数 | ０～２歳３～５歳 | ６～８歳９～11歳(12～14歳) | 男女計 | 1329 | 456 | 213 | 243 | 564 | 268 | 296 | 309 | 男 | 681 | 233 | 109 | 124 | 289 | 137 | 152 | 158 | 女 | 648 | 222 | 104 | 118 | 275 | 131 | 144 | 151 | 10.8 | 3.7 | 1.7 | 2.0 | 4.6 | 2.2 | 2.4 | 2.5 | 人 口 | (万人) | 総人口に占める | 割合(％) | 213 | 243 | 268 | 296 | 309 | 0 | 100 | 200 | 300 | 400 | ０～２ | ３～５ | ６～８ | ９～11 | 12～14 | (万人) | (歳) | 2026年 | 2025年 | ４月１日現在４月１日現在 | 男女計 | 1329 | 1365 | -35 | 男 | 681 | 699 | -18
画像だけの PDF から抽出したテキスト: ''


In [3]:
# 正解の分かる PDF を自分で作る (架空のデータ)。表 1 は数値を書くが、図 1 の折れ線には値を書き込まない
import matplotlib.pyplot as plt
import matplotlib_fontja

plt.rcParams["pdf.fonttype"] = 42          # フォントを TrueType で埋め込む (テキストとして抽出できる)
years = ["2019", "2020", "2021", "2022", "2023"]
table = {"来館者数(千人)": [412, 198, 265, 331, 356], "貸出冊数(千冊)": [1520, 1105, 1283, 1347, 1398], "電子書籍(千件)": [12, 48, 61, 70, 77]}
series = {"一般書": [820, 560, 640, 690, 700], "児童書": [430, 380, 450, 470, 520], "雑誌": [270, 165, 193, 187, 178]}

fig = plt.figure(figsize=(8.27, 11.69))    # A4
fig.text(0.08, 0.95, "架空市立図書館の利用状況 (架空のデータ)", fontsize=16, weight="bold")
fig.text(0.08, 0.91, "2020年度は臨時休館の影響で来館者数が大きく減少した。電子書籍の貸出は毎年増えている。", fontsize=10)
ax = fig.add_axes([0.08, 0.62, 0.84, 0.25]); ax.axis("off"); ax.set_title("表1 年度別の利用状況", fontsize=12)
t = ax.table(cellText=[[y] + [str(v[i]) for v in table.values()] for i, y in enumerate(years)],
             colLabels=["年度"] + list(table), loc="center", cellLoc="center")
t.scale(1, 1.8); t.set_fontsize(11)
ax2 = fig.add_axes([0.12, 0.12, 0.8, 0.38])
for label, v in series.items():
    ax2.plot(years, v, marker="o", label=label)
ax2.set_title("図1 分類別の貸出冊数 (千冊)", fontsize=12); ax2.set_ylim(0, 900); ax2.grid(alpha=0.4); ax2.legend()
fig.savefig("report.pdf"); plt.close(fig)

rep = pymupdf.open("report.pdf")[0]
print(rep.get_text().replace("\n", " | "))  # 表の数値と軸の目盛りは取れるが、折れ線の値はどこにもない

年度 | 来館者数(千人) | 貸出冊数(千冊) | 電子書籍(千件) | 2019 | 412 | 1520 | 12 | 2020 | 198 | 1105 | 48 | 2021 | 265 | 1283 | 61 | 2022 | 331 | 1347 | 70 | 2023 | 356 | 1398 | 77 | 表1 年度別の利用状況 | 2019 | 2020 | 2021 | 2022 | 2023 | 0 | 100 | 200 | 300 | 400 | 500 | 600 | 700 | 800 | 900 | 図1 分類別の貸出冊数 (千冊) | 一般書 | 児童書 | 雑誌 | 架空市立図書館の利用状況 (架空のデータ) | 2020年度は臨時休館の影響で来館者数が大きく減少した。電子書籍の貸出は毎年増えている。 | 


In [4]:
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

name = "Qwen/Qwen3-VL-4B-Instruct"         # Apache-2.0。重みは約 8.9GB (初回はダウンロードに数分かかる)
BF16 = torch.cuda.get_device_capability()[0] >= 8   # bfloat16 は Ampere 以降。T4 では float16
processor = AutoProcessor.from_pretrained(name)
model = AutoModelForImageTextToText.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="cuda").eval()

def page_image(page, dpi):
    pix = page.get_pixmap(dpi=dpi)          # 解像度を dpi で指定して画像にする (72 dpi で 1pt = 1 画素)
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

def ask(question, image=None, text=None, max_new_tokens=64):
    content = []
    if image is not None:
        content.append({"type": "image", "image": image})
    if text is not None:                     # 比較用: 同じモデルに抽出したテキストだけを渡す
        content.append({"type": "text", "text": "次は PDF のページから抽出したテキストです。\n" + text})
    content.append({"type": "text", "text": question})
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_image = int((inputs["input_ids"] == model.config.image_token_id).sum())
    return processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip(), n_image

img = page_image(pub, 100)
md, n_image = ask("このページの内容を Markdown に変換してください。表は Markdown の表にしてください。", image=img, max_new_tokens=1500)
print(img.size, "画素 →", n_image, "画像トークン")
print(md)

(827, 1170) 画素 → 962 画像トークン
```markdown
# 1 こどもの数は1329万人、45年連続の減少

2026年4月1日現在におけるこどもの数（15歳未満人口。以下同じ。）は、前年に比べ35万人少ない1329万人で、1982年から45年連続の減少となり、過去最少となりました。

男女別では、男子が681万人、女子が648万人となっており、男子が女子より33万人多く、女子100人に対する男子の数（人口性比）は105.0となっています。

（表1）

こどもの数を年齢3歳階級別にみると、12～14歳が309万人（総人口に占める割合2.5%）、9～11歳が296万人（同2.4%）、6～8歳が268万人（同2.2%）、3～5歳が243万人（同2.0%）、0～2歳が213万人（同1.7%）となっています。（表2、図1）

これを中学生の年代（12～14歳）、小学生の年代（6～11歳）、未就学の乳幼児（0～5歳）の三つの区分でみると、それぞれ309万人（同割合2.5%）、564万人（同4.6%）、456万人（同3.7%）となっています。

（表2）

表1 男女別こどもの数

| | 2026年4月1日現在 | 2025年4月1日現在 | 対前年増減数 |
| :--- | :--- | :--- | :--- |
| **こどもの数** | | | |
| （万人） | 1329 | 1365 | -35 |
| 男 | 681 | 699 | -18 |
| 女 | 648 | 666 | -17 |
| 人口性比 | 105.0 | 105.0 | 0.0 |
| **総人口** | | | |
| （万人） | 12286 | 12340 | -54 |
| 男 | 5978 | 6004 | -25 |
| 女 | 6307 | 6336 | -29 |
| 人口性比 | 94.8 | 94.7 | 0.1 |
| 総人口に占めるこどもの割合(%) | 10.8 | 11.1 | -0.3 |

注）表中の数値は、単位未満を四捨五入しているため、合計の数値と内訳の計が一致しない場合があります（以下同じ。）。

図1 年齢3歳階級別こどもの数（2026年4月1日現在）

![図1](https://example.com/image.pn

In [5]:
import re

# Markdown への変換では図が画像への参照に置き換わるので、グラフの値は明示的に表にさせる
print(ask("図1の棒グラフの値を読み取って、Markdown の表にしてください。", image=page_image(pub, 100), max_new_tokens=300)[0])

# 自作 PDF の折れ線グラフ (値は書き込んでいない) を表にさせ、正解との差を測る
for dpi in [100, 150]:
    ans = ask("図1の折れ線グラフについて、各分類・各年度の値を読み取って Markdown の表にしてください。",
              image=page_image(rep, dpi), max_new_tokens=400)[0]
    rows = {r[0]: r[1:] for line in ans.splitlines() if re.match(r"\|\s*20\d\d", line)
            for r in [[int(x) for x in re.findall(r"\d+", line)]]}
    if dpi == 100:
        print(ans)
    errors = [abs(rows[int(y)][j] - v[i]) for i, y in enumerate(years) for j, v in enumerate(series.values())]
    print(f"{dpi} dpi: 15 個の値の平均絶対誤差 {sum(errors) / len(errors):.1f} 千冊、最大 {max(errors)} 千冊")

```markdown
| 年齢帯 | 213 | 243 | 268 | 296 | 309 |
|--------|-----|-----|-----|-----|-----|
| 0～2   |     |     |     |     |     |
| 3～5   |     |     |     |     |     |
| 6～8   |     |     |     |     |     |
| 9～11  |     |     |     |     |     |
| 12～14 |     |     |     |     |     |
```
以下は、図1の折れ線グラフから読み取った各分類・各年度の値をMarkdown表でまとめたものです。

| 年度 | 一般書 (千冊) | 児童書 (千冊) | 雑誌 (千冊) |
|------|---------------|---------------|-------------|
| 2019 | 820           | 430           | 270         |
| 2020 | 570           | 380           | 170         |
| 2021 | 640           | 450           | 200         |
| 2022 | 700           | 470           | 190         |
| 2023 | 700           | 520           | 180         |

※ 上記の値はグラフの目盛りと線の位置から推定した値です。グラフの正確な値は、図のデータポイントの座標から読み取ることで確認できます。例えば、2019年の一般書は800に近い値、2020年は約570、2021年は約640、2022年は約700、2023年は約700と推定されます。児童書と雑誌も同様に推定しています。
100 dpi: 15 個の値の平均絶対誤差 2.5 千冊、最大 10 千冊
150 dpi: 15 個の値の平均絶対誤差 3.1 千冊、最大 10 千冊


In [6]:
import unicodedata

QA = [  # (ページ, 質問, 正解)。統計局の資料の正解はページに書かれた値
    ("pub", "表1で、2025年4月1日現在の女のこどもの数は何万人ですか。", "666"),
    ("pub", "表1で、総人口(男女計)の対前年増減数は何万人ですか。", "-54"),
    ("pub", "表2で、9～11歳の男のこどもの数は何万人ですか。", "152"),
    ("pub", "表2で、未就学の乳幼児(0～5歳)の女の数は何万人ですか。", "222"),
    ("pub", "表2で、3～5歳のこどもが総人口に占める割合は何％ですか。", "2.0"),
    ("pub", "図1で、こどもの数が最も多い年齢階級はどれですか。", "12～14"),
    ("rep", "表1で、2021年度の貸出冊数は何千冊ですか。", "1283"),
    ("rep", "表1で、来館者数が最も少ない年度はどれですか。", "2020"),
    ("rep", "図1で、2020年度の一般書の貸出冊数はおよそ何千冊ですか。", "560"),
    ("rep", "図1で、2023年度の児童書の貸出冊数はおよそ何千冊ですか。", "520"),
    ("rep", "図1で、雑誌の貸出冊数が最も多い年度はどれですか。", "2019"),
    ("rep", "図1で、2019年度より2023年度のほうが貸出冊数が多い分類はどれですか。", "児童書"),
]
pages = {"pub": pub, "rep": rep}

def is_correct(answer, gold, question):
    a = unicodedata.normalize("NFKC", answer).replace(",", "").replace("~", "～").replace("−", "-")
    if not re.fullmatch(r"-?[\d.]+", gold):
        return a.startswith(gold)            # 「12～14歳」「2020年度」は可、「一般書、児童書」のような列挙は不可
    nums = re.findall(r"-?\d+(?:\.\d+)?", a)
    if not nums:
        return False
    tol = 0.05 * abs(float(gold)) if "およそ" in question else 0   # グラフから読む値は ±5% まで正解とする
    return abs(float(nums[0]) - float(gold)) <= tol

def evaluate(mode, dpi=100, verbose=True):
    score, wrong = 0, []
    for pg, q, gold in QA:
        q2 = q + "答えだけを短く答えてください。"
        ans = ask(q2, text=pages[pg].get_text())[0] if mode == "text" else ask(q2, image=page_image(pages[pg], dpi))[0]
        ok = is_correct(ans, gold, q); score += ok
        if not ok:
            wrong.append(f"{gold}→{ans[:8]!r}")
        if verbose:
            print(f"{'○' if ok else '×'} 正解 {gold:>6} | {ans[:30]!r} | {q}")
    return score, wrong

for mode in ["text", "image"]:
    print(f"--- {mode}: {evaluate(mode)[0]} / {len(QA)}")

○ 正解    666 | '666' | 表1で、2025年4月1日現在の女のこどもの数は何万人ですか。
○ 正解    -54 | '-54' | 表1で、総人口(男女計)の対前年増減数は何万人ですか。
× 正解    152 | '118' | 表2で、9～11歳の男のこどもの数は何万人ですか。
× 正解    222 | '104' | 表2で、未就学の乳幼児(0～5歳)の女の数は何万人ですか。
○ 正解    2.0 | '2.0％' | 表2で、3～5歳のこどもが総人口に占める割合は何％ですか。
× 正解  12～14 | '6～8歳' | 図1で、こどもの数が最も多い年齢階級はどれですか。
○ 正解   1283 | '1283' | 表1で、2021年度の貸出冊数は何千冊ですか。
○ 正解   2020 | '2020年度' | 表1で、来館者数が最も少ない年度はどれですか。
× 正解    560 | '300' | 図1で、2020年度の一般書の貸出冊数はおよそ何千冊ですか。
× 正解    520 | '300' | 図1で、2023年度の児童書の貸出冊数はおよそ何千冊ですか。
× 正解   2019 | '2023年' | 図1で、雑誌の貸出冊数が最も多い年度はどれですか。
× 正解    児童書 | '一般書、雑誌' | 図1で、2019年度より2023年度のほうが貸出冊数が多い分類はどれですか。
--- text: 5 / 12
○ 正解    666 | '666' | 表1で、2025年4月1日現在の女のこどもの数は何万人ですか。
○ 正解    -54 | '-54' | 表1で、総人口(男女計)の対前年増減数は何万人ですか。
× 正解    152 | '296' | 表2で、9～11歳の男のこどもの数は何万人ですか。
○ 正解    222 | '222' | 表2で、未就学の乳幼児(0～5歳)の女の数は何万人ですか。
× 正解    2.0 | '1.7' | 表2で、3～5歳のこどもが総人口に占める割合は何％ですか。
○ 正解  12～14 | '12～14歳' | 図1で、こどもの数が最も多い年齢階級はどれですか。
○ 正解   1283 | '1283' | 表1で、2021年度の貸出冊数は何千冊ですか。
○ 正解   

In [7]:
import time

# 解像度を変えて、画像トークン数・正答数・1 問あたりの時間・VRAM の最大使用量を比べる
for dpi in [50, 72, 100, 150, 200]:
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.perf_counter()
    score, wrong = evaluate("image", dpi, verbose=False)
    sec = (time.perf_counter() - t) / len(QA)
    n_image = ask("", image=page_image(pub, dpi), max_new_tokens=1)[1]
    print(f"{dpi:3d} dpi: {page_image(pub, dpi).size} 画素, {n_image:4d} 画像トークン, 正答 {score:2d}/{len(QA)}, "
          f"{sec:.2f} 秒/問, 最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print("     誤答 (正解→出力):", ", ".join(wrong))

 50 dpi: (414, 585) 画素,  234 画像トークン, 正答  4/12, 0.20 秒/問, 最大 9.0 GB
     誤答 (正解→出力): 666→'1229', -54→'-0.4', 152→'2.7', 222→'0.8', 2.0→'11.1', 12～14→'0～2歳', 560→'600', 児童書→'一般書'
 72 dpi: (596, 842) 画素,  494 画像トークン, 正答  7/12, 0.26 秒/問, 最大 9.1 GB
     誤答 (正解→出力): 666→'699', -54→'-25', 152→'288', 2.0→'1.7', 12～14→'9～11歳'
100 dpi: (827, 1170) 画素,  962 画像トークン, 正答  9/12, 0.37 秒/問, 最大 9.2 GB
     誤答 (正解→出力): 152→'296', 2.0→'1.7', 児童書→'一般書'
150 dpi: (1241, 1755) 画素, 2145 画像トークン, 正答 10/12, 0.75 秒/問, 最大 9.5 GB
     誤答 (正解→出力): 666→'699', 222→'104'
200 dpi: (1654, 2339) 画素, 3796 画像トークン, 正答 10/12, 1.30 秒/問, 最大 10.0 GB
     誤答 (正解→出力): 152→'131', 児童書→'一般書'


- 出力の値は float16（T4 と同じ）で実行したもの。bfloat16（Ampere 以降の GPU）でも大部分は同じで、違うところは括弧内に書く。
- テキストを抽出すると、表 2 の 2 段の見出しが「こどもの未就学の乳幼児（０～５歳）| 小学生（６～11歳）| 中学生 | 数 | …」のように崩れ、どの列の値かが分からなくなる。同じモデルにこのテキストだけを渡すと、表 2 の「9～11 歳の男」を 118、「0～5 歳の女」を 104 と答えた（正解は 152、222）。図 1 の棒グラフの値 (213〜309) はテキストに含まれているのに、最も多い年齢階級を「6～8歳」と答えた。一方、自作 PDF の表 1 のような見出しが 1 段の表は、テキストだけでも正しく答えた。失われるのは、入れ子の見出しやグラフのように位置で意味が決まる情報である。
- 値を書き込んでいない折れ線グラフは、テキストには軸の目盛りしか残らない。テキストだけでは 2020 年度の一般書も 2023 年度の児童書も「300」と答えた（正解は 560、520）。画像を渡すと 550 と 520（bfloat16 では 570 と 520）になった。グラフ全体を表にさせると、15 個の値の誤差は 100 dpi で平均 2.5 千冊（bfloat16 では 4.2 千冊）、最大 10 千冊だった。グラフから読んだ値は目盛りの細かさに応じた誤差を含む推定値で、元のデータの代わりにはならない。
- 画像を渡しても誤りはある。表 2 の「9～11 歳の男」には男女計の行の 296 を、「3～5 歳の割合」には 1 列左の 1.7 を答えた。Markdown に変換した表 2 も、2 段目の見出しが 7 列しかなく、9 列のデータとずれている。図 1 は `![図1](https://example.com/image.png)` という実在しない URL への参照に置き換わり、値を表にさせると値が列見出しに入った空の表になった。出力はもっともらしい形をしていても、元のページと突き合わせて確かめる。結合したセルのある表は、セルごとに質問するなどの工夫が要る。
- この例では、「2019 年度より 2023 年度のほうが多い分類」（正解は児童書）に 50、100、200 dpi で「一般書」と答えた。1 つの値を読む質問より間違えやすかった。
- 画像トークン数は画素数にほぼ比例する（約 32×32 画素で 1 トークン）。A4 のページで 50 dpi なら 234、100 dpi なら 962、200 dpi なら 3796 だった。正答は 50 dpi で 4/12、72 dpi で 7/12、100 dpi で 9/12、150 dpi と 200 dpi で 10/12。1 問あたりの時間は 0.20 秒から 1.30 秒に、VRAM の最大使用量は 9.0GB から 10.0GB に増えた（重みだけで約 8.9GB）。T4 (16GB) に収まる。時間は検証の GPU で測ったもので、T4 では長くなると見込まれる（T4 では測っていない）。このページでは 100〜150 dpi が釣り合いがよい。bfloat16 では 150 dpi の正答が 9/12 だった。数値の精度が違うだけで 1 問変わるので、12 問での 1 問の差は誤差の範囲である。
- スキャンした PDF にはテキスト層がなく、`get_text()` は空文字列を返す（上の画像だけの PDF）。ページを画像にして VLM に渡すか、この章の「画像の中の日本語を読み取り (OCR)、文字誤り率で比べる」の方法を使う。長い文書はページごとに処理する。100 dpi でも 1 ページで約 1000 トークンになる。ページごとの Markdown を RAG の章の「長い文書をチャンクに分割する」で分割すれば、RAG に使える。ページの画像のまま検索する方法は、この章の「PDF のページを画像のまま検索して VLM に答えさせる（マルチモーダル RAG）」を参照。
- Qwen3-VL-4B-Instruct のライセンスは Apache-2.0（モデルカード）。古い transformers では読み込めない（検証時の Colab の transformers 5.22.1 では動いた）。モデルカードはサンプリング（temperature=0.7 など）を推奨しているが、ここでは再現性のために貪欲法 (`do_sample=False`) にした。